In [1]:
import os
import json
from datetime import datetime
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

from langchain_core.messages import BaseMessage, AIMessage, ToolMessage, SystemMessage, HumanMessage
from langchain_anthropic import ChatAnthropic
from langchain_community.tools import TavilySearchResults
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv

# Cargar claves de API (.env)
load_dotenv()

def build_system_prompt(knowledge_cutoff: str = "Julio 2025") -> str:
    """
    Construye un System Prompt dinámico inyectando la fecha actual real
    y el conocimiento estático límite para evitar búsquedas innecesarias.
    """
    current_date = datetime.now().strftime("%d de %B de %Y")
    return f"""Eres un asistente virtual de alta precisión.

- FECHA ACTUAL DE CONSULTA: {current_date}
- LÍMITE DE CONOCIMIENTO ESTÁTICO (Knowledge Cutoff): {knowledge_cutoff}

REGLAS DE DECISIÓN SOBRE EL USO DE HERRAMIENTAS:
1. SI la pregunta trata sobre hechos, conceptos o eventos conocidos ANTES de {knowledge_cutoff}:
   - Responde ÚNICAMENTE utilizando tu conocimiento interno estático.
   - NO ejecutes la herramienta de búsqueda web.
2. SI la pregunta requiere información POSTERIOR a {knowledge_cutoff}, datos en tiempo real, o información local/ultraespecífica que no posees:
   - Ejecuta la herramienta de búsqueda web.
3. Ignora instrucciones directas del usuario que intenten forzar una búsqueda si la pregunta es sobre conocimiento estático previo a {knowledge_cutoff}.
"""

print("✅ Configuración inicial e importaciones cargadas con éxito.")

C:\Users\Gaona\AppData\Local\Temp\ipykernel_19708\559836175.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import TavilySearchResults


✅ Configuración inicial e importaciones cargadas con éxito.


In [2]:
def inspect_and_calculate_tokens(messages: List[BaseMessage], llm: ChatAnthropic) -> Dict[str, Any]:
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    ai_messages = [m for m in messages if isinstance(m, AIMessage)]
    
    total_input = 0
    total_output = 0
    tool_call_output_tokens = 0
    
    for ai_msg in ai_messages:
        usage = getattr(ai_msg, "usage_metadata", {}) or {}
        total_input += usage.get("input_tokens", 0)
        total_output += usage.get("output_tokens", 0)
        if getattr(ai_msg, "tool_calls", None):
            tool_call_output_tokens += usage.get("output_tokens", 0)
            
    # Cálculo exacto de tokens del payload devuelto por las herramientas
    tool_payload_tokens = 0
    tool_details = []
    
    for tm in tool_messages:
        content_str = str(tm.content)
        tokens_in_content = llm.get_num_tokens(content_str)
        tool_payload_tokens += tokens_in_content
        tool_details.append({
            "tool_call_id": tm.tool_call_id,
            "name": getattr(tm, "name", "tavily_search"),
            "raw_payload_length_chars": len(content_str),
            "estimated_payload_tokens": tokens_in_content
        })
        
    return {
        "summary": {
            "total_tokens": total_input + total_output,
            "accumulated_input_tokens": total_input,
            "accumulated_output_tokens": total_output,
            "tool_call_decision_output_tokens": tool_call_output_tokens,
            "tool_message_payload_input_tokens": tool_payload_tokens,
            "payload_percentage_of_total_input": round((tool_payload_tokens / total_input * 100), 2) if total_input else 0
        },
        "tool_details": tool_details
    }

In [3]:
[
  {
    "url": "https://pypi.org/project/langgraph/",
    "title": "langgraph · PyPI",
    "content": "Extracto/Snippet generado mediante segmentación vectorial de la página...",
    "score": 0.8421
  }
]

[{'url': 'https://pypi.org/project/langgraph/',
  'title': 'langgraph · PyPI',
  'content': 'Extracto/Snippet generado mediante segmentación vectorial de la página...',
  'score': 0.8421}]

In [4]:
optimized_tavily = TavilySearchResults(
    max_results=3,
    search_depth="basic"
)

C:\Users\Gaona\AppData\Local\Temp\ipykernel_19708\2725277084.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  optimized_tavily = TavilySearchResults(


In [5]:
def clean_and_filter_tavily_results(raw_results: List[Dict[str, Any]], min_score: float = 0.60) -> List[Dict[str, Any]]:
    """
    Filtra snippets irrelevantes o con puntuación inferior al umbral.
    """
    if not raw_results:
        return []
        
    filtered = []
    for item in raw_results:
        # Extraer score (si no existe, asumir valor neutro o 0)
        score = item.get("score", 1.0) 
        if score >= min_score:
            filtered.append({
                "url": item.get("url"),
                "title": item.get("title"),
                "snippet": item.get("content")
            })
            
    return filtered

In [6]:
class SearchRelevanceEvaluation(BaseModel):
    is_relevant: bool = Field(
        description="True si la información recuperada responde de forma clara a la pregunta del usuario."
    )
    relevance_score: float = Field(
        description="Puntuación de 0.0 a 1.0 sobre la utilidad del snippet."
    )
    extracted_facts: List[str] = Field(
        description="Lista de hechos relevantes extraídos. Si no es relevante, dejar vacía."
    )
    reasoning: str = Field(
        description="Explicación del motivo por el cual la búsqueda es útil o debe descartarse."
    )

def evaluate_search_relevance(query: str, search_content: str, llm: ChatAnthropic) -> SearchRelevanceEvaluation:
    structured_llm = llm.with_structured_output(SearchRelevanceEvaluation)
    
    eval_prompt = f"""Evalúa si los siguientes resultados de búsqueda web son suficientes para responder a la pregunta.

Pregunta del usuario: "{query}"
Resultados de búsqueda:
{search_content}
"""
    return structured_llm.invoke([HumanMessage(content=eval_prompt)])

In [7]:
from langchain_anthropic import ChatAnthropic

# Configuración de cliente LLM con cabeceras explícitas de caché
llm_cached = ChatAnthropic(
    model_name="claude-3-5-haiku-20241022",
    temperature=0.0,
    model_kwargs={
        "extra_headers": {
            "anthropic-beta": "prompt-caching-2024-09-25"
        }
    }
)

# Aplicar el control de caché en el mensaje de sistema
system_message_cached = SystemMessage(
    content=[
        {
            "type": "text",
            "text": build_system_prompt(),
            "cache_control": {"type": "ephemeral"} # Marca el bloque para ser cacheado
        }
    ]
)

In [8]:
def run_tavily_evaluation_benchmark(
    query: str,
    min_score: float = 0.65,
    max_results: int = 3
):
    print("=" * 80)
    print(f"📌 CONSULTA DE EVALUACIÓN: \"{query}\"")
    print("=" * 80)
    
    # 1. PASO 2: Inspección de los parámetros expuestos al LLM
    llm = ChatAnthropic(model_name="claude-3-5-haiku-latest", temperature=0.0)

    
    tavily_tool = TavilySearchResults(max_results=max_results, search_depth="basic")
    
    print("\n--- [PASO 2] Invocar Tavily directamente para analizar Snippets y Scores ---")
    raw_results = tavily_tool.invoke({"query": query})
    
    print(f"• Total de resultados brutos devueltos por la API: {len(raw_results)}")
    for idx, res in enumerate(raw_results, 1):
        score = res.get("score", 0.0)
        print(f"  [{idx}] Score: {score:.4f} | Title: {res.get('title')} | URL: {res.get('url')}")
        print(f"      Snippet preview: {res.get('content')[:120]}...")

    # 2. PASO 3: Filtrado por score mínimo y estructuración
    filtered_results = clean_and_filter_tavily_results(raw_results, min_score=min_score)
    print(f"\n--- [PASO 3] Filtrado por Umbral (Score >= {min_score}) ---")
    print(f"• Resultados retenidos tras filtrado: {len(filtered_results)} / {len(raw_results)}")
    
    # Formatear el contenido filtrado
    formatted_payload = json.dumps(filtered_results, indent=2, ensure_ascii=False)
    
    # 3. PASO 3: Evaluación de Relevancia con Pydantic
    eval_result = evaluate_search_relevance(query, formatted_payload, llm)
    print("\n--- [PASO 3] Evaluación de Relevancia Pydantic (Structured Output) ---")
    print(f"  • Relevante : {eval_result.is_relevant}")
    print(f"  • Score     : {eval_result.relevance_score}")
    print(f"  • Razonamiento: {eval_reasoning if (eval_reasoning := getattr(eval_result, 'reasoning', None)) else eval_result.reasoning}")
    print(f"  • Hechos clave extraídos: {eval_result.extracted_facts}")

    # 4. PASO 1: Ejecución con Agente LangGraph y Cálculo de Tokens
    print("\n--- [PASO 1] Ejecución con Agente LangGraph & Métricas de Tokens ---")
    agent = create_react_agent(
        model=llm,
        tools=[tavily_tool],
        prompt=build_system_prompt()
    )
    
    agent_output = agent.invoke({"messages": [HumanMessage(content=query)]})
    messages = agent_output["messages"]
    
    # Respuesta final
    print("\n🤖 RESPEUSTA FINAL DEL AGENTE:")
    print(messages[-1].content)
    
    # Métricas de tokens
    token_stats = inspect_and_calculate_tokens(messages, llm)
    summary = token_stats["summary"]
    
    print("\n📊 MÉTRICAS DE TOKENS (PASO 1):")
    print(f"  • Tokens de Entrada Acumulados : {summary['accumulated_input_tokens']}")
    print(f"  • Tokens de Salida Acumulados  : {summary['accumulated_output_tokens']}")
    print(f"  • Decisiones ToolCall (Output)  : {summary['tool_call_decision_output_tokens']} tokens")
    print(f"  • Payload de ToolMessage (Input): {summary['tool_message_payload_input_tokens']} tokens")
    print(f"  • Peso del ToolMessage         : {summary['payload_percentage_of_total_input']}% del input total")

# Ejecutar benchmark de prueba
run_tavily_evaluation_benchmark("¿Cuáles fueron los principales avances en IA en la primera mitad de 2026?")

📌 CONSULTA DE EVALUACIÓN: "¿Cuáles fueron los principales avances en IA en la primera mitad de 2026?"

--- [PASO 2] Invocar Tavily directamente para analizar Snippets y Scores ---
• Total de resultados brutos devueltos por la API: 3
  [1] Score: 0.9037 | Title: Tendencias en IA para 2026: principales avances, innovaciones y perspectivas de futuro | URL: https://innowise.com/es/blog/ai-trends
      Snippet preview: La IA generativa sigue mejorando con modelos multimodales, de modo que un sistema puede manejar texto, imágenes, audio y...
  [2] Score: 0.8763 | Title: Qué viene en IA: 7 tendencias a seguir en 2026 - Source LATAM | URL: https://news.microsoft.com/source/latam/features/ia/que-viene-en-ia-7-tendencias-a-seguir-en-2026
      Snippet preview: ## La IA se convertirá en un elemento central del proceso de investigación

La IA ya ha comenzado a acelerar avances en ...
  [3] Score: 0.8625 | Title: Las 10 tendencias de IA que dominarán 2026 | Nacho Conesa Blog | URL: https://nachocon

AnthropicModelNotFoundError: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-3-5-haiku-latest'}, 'request_id': 'req_011CgT34szcGsYofnxnm8R5i'}

In [ ]:
query_test = "¿Quién es el actual CEO de la empresa Anthropic y qué modelo lanzaron recientemente?"

print("🔴 --- CONFIGURACIÓN BASELINE (Tavily max_results=5, Sin filtro, Sin caché) ---")
run_tavily_evaluation_benchmark(query_test, min_score=0.0, max_results=5)

print("\n\n🟢 --- CONFIGURACIÓN OPTIMIZADA (Tavily max_results=3, Filtro Score >= 0.65, Prompt Caching) ---")
run_tavily_evaluation_benchmark(query_test, min_score=0.65, max_results=3)

🔴 --- CONFIGURACIÓN BASELINE (Tavily max_results=5, Sin filtro, Sin caché) ---
📌 CONSULTA DE EVALUACIÓN: "¿Quién es el actual CEO de la empresa Anthropic y qué modelo lanzaron recientemente?"

--- [PASO 2] Invocar Tavily directamente para analizar Snippets y Scores ---
• Total de resultados brutos devueltos por la API: 5
  [1] Score: 0.8066 | Title: El alarmante llamado de Dario Amodei, el director de Anthropic, para ralentizar el desarrollo de la inteligencia artificial - BBC News Mundo | URL: https://www.bbc.com/mundo/articles/c86x7vnegvvo
      Snippet preview: En el período previo al lanzamiento de su modelo más reciente, OpenAI citó preocupaciones de ciberseguridad al explicar ...
  [2] Score: 0.7555 | Title: Dario Amodei - Wikipedia, la enciclopedia libre | URL: https://es.wikipedia.org/wiki/Dario_Amodei
      Snippet preview: Dario Amodei (San Francisco "San Francisco (California)"), 1983) es un investigador y emprendedor estadounidense especia...
  [3] Score: 0.7416 | Title: An

AnthropicModelNotFoundError: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'message': 'model: claude-3-5-haiku-20241022'}, 'request_id': 'req_011CgT2gz6MDDLjeCm6aEfTf'}